In [1]:
!git clone https://github.com/namkuner/Tran_PL-BERT_VietNamese.git

In [8]:
!pip install datasets
!pip install accelerate
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 884.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 55.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.5/776.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 12.0 MB/s eta 0:00:0000:0100:01


In [1]:
import os
os.chdir('/workspace/Tran_PL-BERT_VietNamese')

In [2]:
import os
import shutil
import os.path as osp

import torch
from torch import nn
import torch.nn.functional as F

from accelerate import Accelerator
from accelerate.utils import LoggerType

from transformers import AdamW
from transformers import AlbertConfig, AlbertModel
from accelerate import DistributedDataParallelKwargs

from model import MultiTaskModel
from dataloader import build_dataloader
from utils import length_to_mask, scan_checkpoint

from datasets import load_dataset


In [3]:

import yaml
import pickle

config_path = "Configs/config.yml" # you can change it to anything else
config = yaml.safe_load(open(config_path))

In [4]:
import pickle

with open(config['dataset_params']['token_maps'], 'rb') as handle:
    token_maps = pickle.load(handle)

In [5]:
criterion = nn.CrossEntropyLoss() # F0 loss (regression)

best_loss = float('inf')  # best test loss
start_epoch = 0  # start from epoch 0 or last checkpoint epoch
loss_train_record = list([])
loss_test_record = list([])

num_steps = config['num_steps']
log_interval = config['log_interval']
save_interval = config['save_interval']

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [7]:
def train():

    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)

    curr_steps = 0

    dataset = load_dataset("namkuner/phonemes_vi")
    dataset = dataset['train']

    log_dir = config['log_dir']
    if not osp.exists(log_dir): os.makedirs(log_dir, exist_ok=True)
    shutil.copy(config_path, osp.join(log_dir, osp.basename(config_path)))

    batch_size = config["batch_size"]
    train_loader = build_dataloader(dataset,
                                    batch_size=batch_size,
                                    num_workers=0,
                                    dataset_config=config['dataset_params'])

    albert_base_configuration = AlbertConfig(**config['model_params'])

    bert = AlbertModel(albert_base_configuration)
    bert = MultiTaskModel(bert,
                          num_vocab=1 + max([m['token'] for m in token_maps.values()]),
                          num_tokens=config['model_params']['vocab_size'],
                          hidden_size=config['model_params']['hidden_size'])

    load = True
    try:
        files = os.listdir(log_dir)
        ckpts = []
        for f in os.listdir(log_dir):
            if f.startswith("step_"): ckpts.append(f)

        iters = [int(f.split('_')[-1].split('.')[0]) for f in ckpts if os.path.isfile(os.path.join(log_dir, f))]
        iters = sorted(iters)[-1]
    except:
        iters = 0
        load = False

    optimizer = AdamW(bert.parameters(), lr=1e-4)

    accelerator = Accelerator(mixed_precision=config['mixed_precision'], split_batches=True, kwargs_handlers=[ddp_kwargs])

    if load:
        checkpoint = torch.load(log_dir + "/step_" + str(iters) + ".t7", map_location='cpu')
        state_dict = checkpoint['net']
        from collections import OrderedDict
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:] # remove `module.`
            new_state_dict[name] = v

        bert.load_state_dict(new_state_dict, strict=False)

        accelerator.print('Checkpoint loaded.')
        optimizer.load_state_dict(checkpoint['optimizer'])

    bert, optimizer, train_loader = accelerator.prepare(
        bert, optimizer, train_loader
    )

    accelerator.print('Start training...')

    running_loss = 0

    for _, batch in enumerate(train_loader):
        curr_steps += 1

        words, labels, phonemes, input_lengths, masked_indices = batch
        text_mask = length_to_mask(torch.Tensor(input_lengths)).to(device)

        tokens_pred, words_pred = bert(phonemes, attention_mask=(~text_mask).int())

        loss_vocab = 0
        for _s2s_pred, _text_input, _text_length, _masked_indices in zip(words_pred, words, input_lengths, masked_indices):
            loss_vocab += criterion(_s2s_pred[:_text_length],
                                        _text_input[:_text_length])
        loss_vocab /= words.size(0)

        loss_token = 0
        sizes = 1
        for _s2s_pred, _text_input, _text_length, _masked_indices in zip(tokens_pred, labels, input_lengths, masked_indices):
            if len(_masked_indices) > 0:
                _text_input = _text_input[:_text_length][_masked_indices]
                loss_tmp = criterion(_s2s_pred[:_text_length][_masked_indices],
                                            _text_input[:_text_length])
                loss_token += loss_tmp
                sizes += 1
        loss_token /= sizes

        loss = loss_vocab + loss_token

        optimizer.zero_grad()
        accelerator.backward(loss)
        optimizer.step()

        running_loss += loss.item()

        iters = iters + 1
        if (iters+1)%log_interval == 0:
            accelerator.print ('Step [%d/%d], Loss: %.5f, Vocab Loss: %.5f, Token Loss: %.5f'
                    %(iters+1, num_steps, running_loss / log_interval, loss_vocab, loss_token))
            running_loss = 0

        if (iters+1)%save_interval == 0:
            accelerator.print('Saving..')

            state = {
                'net':  bert.state_dict(),
                'step': iters,
                'optimizer': optimizer.state_dict(),
            }

            accelerator.save(state, log_dir + '/step_' + str(iters + 1) + '.t7')

        if curr_steps > num_steps:
            return

In [ ]:
from accelerate import notebook_launcher
import torch
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

while True:
    notebook_launcher(train, args=(), num_processes=1, use_port=33389)


Launching training on one GPU.


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

186


/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:457: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(split_batches=True)
  warnings.warn(


Start training...
Step [10/1000000], Loss: 11.97165, Vocab Loss: 9.03969, Token Loss: 3.46566
Step [20/1000000], Loss: 11.87215, Vocab Loss: 8.19189, Token Loss: 3.17077
Step [30/1000000], Loss: 10.89206, Vocab Loss: 7.43296, Token Loss: 3.19164
Step [40/1000000], Loss: 10.26682, Vocab Loss: 6.83411, Token Loss: 3.12392
Step [50/1000000], Loss: 9.79063, Vocab Loss: 6.43583, Token Loss: 3.10793
Step [60/1000000], Loss: 9.51514, Vocab Loss: 6.61926, Token Loss: 3.10161
Step [70/1000000], Loss: 9.34271, Vocab Loss: 6.12403, Token Loss: 3.10670
Step [80/1000000], Loss: 9.09908, Vocab Loss: 5.85005, Token Loss: 3.06996
Step [90/1000000], Loss: 9.12621, Vocab Loss: 6.18953, Token Loss: 3.11967
Step [100/1000000], Loss: 9.10895, Vocab Loss: 6.00452, Token Loss: 3.20534
Step [110/1000000], Loss: 8.98437, Vocab Loss: 5.95263, Token Loss: 3.03741
Step [120/1000000], Loss: 8.94147, Vocab Loss: 5.93825, Token Loss: 3.09631
Step [130/1000000], Loss: 8.95194, Vocab Loss: 5.82424, Token Loss: 3.06780